In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/cicidscollection/cic-collection.parquet


# Improving the CIC collection

## 1. By removing contaminating features

Features have been found in the CIC datasets and in other NIDS datasets which have equal, blind predictive power across all available attack classes, despite having had access to only one attack class during training.

Those features have content which the algorithms easily find and exploit to create models that have good to excellent performance, but which have little or nothing to do with the actual attacks in the dataset. 

I have [presented these findings](http://dx.doi.org/10.13140/RG.2.2.23883.46886) at the 2022 edition of the conference for privacy, security and trust (PST). The conference is organized by the Canadian Institute for Cybersecurity. I highly recommend reading the slides and the notes pages for a complete understanding of the methodology and results. [The conference article](http://dx.doi.org/10.1109/PST55820.2022.9851974) contains largely the same information, but the results were expanded further between the acceptance of the article and its presentation. Therefore the presentation is the go-to for a complete and visualized summary.

In this notebook, I will remove those features that have been found to be contaminants in the CIC collection (IDS17, DoS17, IDS18, DDoS19), resulting in the second version of [this dataset here on Kaggle](https://www.kaggle.com/datasets/dhoogla/cicidscollection).

The features which contaminate the CIC collection dataset in the aforementioned way are in order of severity:

* PSH Flag Count, ECE Flag Count, RST Flag Count, ACK Flag Count
* Fwd Packet Length Min
* Bwd Packet Length Min
* Packet Length Min
* Protocol

To a lesser degree *Down/Up Ratio*, *Active Std* and *Idle Std* are also affected, but only the first one will be removed. For the remaining two, the contaminating influence was not consistent enough between algorithms, nor pronounced enough.

In [10]:
df = pd.read_parquet('/kaggle/input/cicidscollection/cic-collection.parquet')
df.shape

(9167581, 79)

In [11]:
df = df.drop(columns=['PSH Flag Count', 'ECE Flag Count', 'RST Flag Count', 'ACK Flag Count', 'Fwd Packet Length Min', 'Bwd Packet Length Min', 'Packet Length Min', 'Protocol', 'Down/Up Ratio'])
df.shape

(9167581, 70)

## 2. By removing features with no separating power
During the analysis to find contaminating features, other features have been found in the CIC collection and in other NIDS datasets, which never contribute meaningfully (or even at all) in any classification model. 

For the CIC collection, 11 features with 0 predictive power have been identified.

In [12]:
df.columns

Index(['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
       'Fwd Packets Length Total', 'Bwd Packets Length Total',
       'Fwd Packet Length Max', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Max', 'Packet Length Mean', 'Packet Length Std',
       'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count',
       'URG Flag Count', 'CWE Flag Count', 'Avg Packet Size',
       'Avg Fwd Segment Size', 'Avg Bwd Segment Si

In [13]:
df = df.drop(columns=['Bwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd PSH Flags', 'Bwd URG Flags', 'CWE Flag Count', 'FIN Flag Count', 'Fwd Avg Bulk Rate', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd URG Flags'])
df.shape

(9167581, 59)

In [14]:
df.to_parquet('cic-collection.parquet')